# ROCM-QLORA GPU Test Notebook

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy any errors back to the chat

In [ ]:
# Cell 1: Setup
!git clone https://github.com/Bala-Mosay/ROCM-QLORA.git
%cd ROCM-QLORA
!pip install -e ".[dev]"

In [ ]:
# Cell 2: GPU Info
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")
else:
    print("WARNING: No GPU detected!")

In [ ]:
# Cell 3: Full Test Suite (212 tests)
!python -m pytest tests/ -v --tb=short

In [ ]:
# Cell 4: Smoke Tests
!python smoke_test.py
!python smoke_test_v3.py
!python smoke_test_v4.py
!python smoke_test_v5.py

In [ ]:
# Cell 5: GPU Training Test
!python -c "
import torch
import torch.nn as nn
from rocm_qlora import quantize_model, enable_all_kernels

model = nn.Sequential(nn.Linear(128, 128), nn.Linear(128, 10))
model = quantize_model(model, bits=4, lora_r=4, target_modules=['0'])
n = enable_all_kernels(model)
print(f'Enabled kernels: {n}')

device = torch.device('cuda')
model = model.to(device)
x = torch.randn(2, 128, device=device)
out = model(x)
loss = out.sum()
loss.backward()
print('GPU training test: PASSED')
"

In [ ]:
# Cell 6: Merge/Unmerge Test
!python -c "
import torch
import torch.nn as nn
from rocm_qlora import quantize_model

model = nn.Sequential(nn.Linear(128, 128), nn.Linear(128, 10))
model = quantize_model(model, bits=4, lora_r=4, target_modules=['0'])

x = torch.randn(1, 128)

# Forward before merge
out_before = model(x)
print(f'Output before merge: {out_before.shape}')

# Merge
model[0].merge_lora()
out_merged = model(x)
diff = (out_before - out_merged).abs().max().item()
print(f'After merge (atol): {diff:.6f}')

# Unmerge
model[0].unmerge_lora()
out_unmerged = model(x)
diff2 = (out_before - out_unmerged).abs().max().item()
print(f'After unmerge (atol): {diff2:.6f}')
print('Merge/Unmerge test: PASSED' if diff2 < 0.01 else 'FAILED')
"

In [ ]:
# Cell 7: Benchmark
!python -c "
from rocm_qlora.hip_kernels import benchmark_hip_vs_triton
result = benchmark_hip_vs_triton()
for k, v in result.items():
    print(f'{k}: {v}')
"

In [ ]:
# Cell 8: Export Test
!python -c "
import torch
import torch.nn as nn
import os
from rocm_qlora import quantize_model
from rocm_qlora.export.gguf_export import merge_and_export_fp16, get_gguf_conversion_instructions

model = nn.Sequential(nn.Linear(128, 128), nn.Linear(128, 10))
model = quantize_model(model, bits=4, lora_r=4, target_modules=['0'])

os.makedirs('/tmp/export_test', exist_ok=True)
path = merge_and_export_fp16(model, '/tmp/export_test/model_fp16.pt')
print(f'Exported to: {path}')
print(f'File size: {os.path.getsize(path) / 1e6:.2f} MB')
print('Export test: PASSED')
"